## Simulação de cenário para redução de churn
--------------------------------------

Simula o IMPACTO POTENCIAL de ações de retenção sobre a taxa de churn, usando o modelo preditivo já treinado como "motor de simulação".

COMO FUNCIONA (metodologia):
  1. Treina o modelo normalmente com os dados históricos.
  2. Calcula a taxa de churn ESPERADA hoje para a base de clientes ativos (a probabilidade média prevista pelo modelo = taxa de churn esperada).
  3. Para cada cenário de ação (ex.: "migrar clientes de contrato mensal para anual"), pega os clientes ativos elegíveis, MUDA artificialmente esse atributo no cadastro deles (ex.: Contract = 'One year') e pede ao modelo treinado para prever a nova probabilidade de churn de cada um — mantendo todos os outros atributos iguais.
  4. Recalcula a taxa de churn esperada da base inteira com esses valores simulados e compara com a taxa original.

RESULTADO: "a taxa de churn esperada caiu de X% para Y%" para cada cenário.

⚠️ IMPORTANTE — O QUE ISSO PROVA (E O QUE NÃO PROVA):
  Isso é uma simulação estatística baseada em CORRELAÇÕES históricas do modelo, não um experimento controlado. Ela responde: "com base nos padrões que o modelo aprendeu, clientes com este outro perfil tendem a cancelar menos". Isso é uma evidência forte e útil para priorizar ações, mas não é prova causal definitiva — outros fatores não observados podem influenciar o resultado real. A forma de transformar isso em prova real é rodar a ação com um grupo piloto de clientes e medir o resultado de fato (teste A/B), como sugerido nos próximos passos do projeto.

COMO RODAR:
    pip install pandas numpy scikit-learn matplotlib  
    python simular_reducao_churn.py

In [15]:
import os
import sys

# Corrige um problema comum no Windows: o console não consegue exibir
# certos caracteres especiais (setas, emojis) e o script quebra com
# "UnicodeEncodeError". Isso força a saída do terminal para UTF-8.
try:
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
except Exception:
    pass

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)


In [16]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================
# Caminho da base de dados: a pasta "data/" fica um nível acima de "notebooks/"
CAMINHO_DADOS = '../data/Telecom_Churn.csv'
SAIDA_CSV = 'simulacao_impacto_cenarios.csv'
SAIDA_GRAFICO = 'simulacao_impacto_cenarios.png'


def carregar_dados_com_verificacao(caminho):
    if not os.path.isfile(caminho):
        raise FileNotFoundError(
            f"\nArquivo não encontrado: '{caminho}'\n"
            f"Pasta atual: {os.getcwd()}\n"
            f"Dica: confirme se está rodando a partir da pasta 'notebooks/', "
            f"ou ajuste CAMINHO_DADOS no início do script."
        )
    return caminho


In [17]:



# ============================================================
# 1) CARREGAR DADOS E TREINAR O MODELO (igual ao pipeline original)
# ============================================================
def carregar_dados(caminho):
    df = pd.read_csv(caminho)
    df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    df['Churn'] = df['Churn'].replace({0: 'No', 1: 'Yes'})
    return df


def preparar_features(df, colunas_referencia=None):
    """One-hot encoding + normalização. Se colunas_referencia for passado,
    garante que o resultado tenha exatamente as mesmas colunas (na mesma
    ordem) que o conjunto usado no treino — necessário para simular."""
    X = df.drop(columns=['customerID', 'Churn'], errors='ignore')
    X = pd.get_dummies(X)
    if colunas_referencia is not None:
        X = X.reindex(columns=colunas_referencia, fill_value=0)
    return X


def treinar_modelo(df):
    X = preparar_features(df)
    colunas_referencia = list(X.columns)

    le = LabelEncoder()
    y = le.fit_transform(df['Churn'])  # No -> 0, Yes -> 1

    mm = MinMaxScaler()
    mm.fit(X)  # guardamos o scaler já ajustado para reutilizar na simulação
    X_scaled = pd.DataFrame(mm.transform(X), columns=colunas_referencia)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.25, random_state=42, stratify=y
    )

    modelo = RandomForestClassifier(max_depth=9, n_estimators=100, random_state=42)
    modelo.fit(X_train, y_train)

    acuracia = modelo.score(X_test, y_test)
    print(f"Acurácia do modelo no conjunto de teste: {acuracia:.2%}")

    return modelo, mm, colunas_referencia


def prever_probabilidades(df, modelo, mm, colunas_referencia):
    X = preparar_features(df, colunas_referencia)
    X_scaled = pd.DataFrame(mm.transform(X), columns=colunas_referencia)
    return modelo.predict_proba(X_scaled)[:, 1]



In [18]:

# ============================================================
# 2) DEFINIÇÃO DOS CENÁRIOS DE AÇÃO (baseados nos insights do projeto)
# ============================================================
def cenario_migrar_contrato_anual(df_ativos):
    """Migra clientes com contrato mensal para contrato anual."""
    elegiveis = df_ativos['Contract'] == 'Month-to-month'
    df_sim = df_ativos.copy()
    df_sim.loc[elegiveis, 'Contract'] = 'One year'
    return df_sim, elegiveis


def cenario_trocar_forma_pagamento(df_ativos):
    """Migra clientes que pagam por cheque eletrônico para débito automático."""
    elegiveis = df_ativos['PaymentMethod'] == 'Electronic check'
    df_sim = df_ativos.copy()
    df_sim.loc[elegiveis, 'PaymentMethod'] = 'Bank transfer (automatic)'
    return df_sim, elegiveis


def cenario_desconto_fibra(df_ativos, percentual_desconto=0.15):
    """Aplica desconto de retenção na mensalidade de clientes com fibra óptica."""
    elegiveis = df_ativos['InternetService'] == 'Fiber optic'
    df_sim = df_ativos.copy()
    df_sim.loc[elegiveis, 'MonthlyCharges'] = (
        df_sim.loc[elegiveis, 'MonthlyCharges'] * (1 - percentual_desconto)
    )
    return df_sim, elegiveis


def cenario_combinado_alto_risco(df_ativos, probas_atuais, limiar_alto_risco=0.50):
    """Nos clientes de ALTO risco: migra para contrato anual E troca a forma
    de pagamento para débito automático, simultaneamente."""
    elegiveis = probas_atuais >= limiar_alto_risco
    df_sim = df_ativos.copy()
    df_sim.loc[elegiveis, 'Contract'] = 'One year'
    df_sim.loc[elegiveis, 'PaymentMethod'] = 'Bank transfer (automatic)'
    return df_sim, elegiveis

In [19]:



# ============================================================
# 3) MOTOR DE SIMULAÇÃO
# ============================================================
def simular_cenario(nome, df_ativos, probas_atuais, df_sim, elegiveis,
                     modelo, mm, colunas_referencia):
    elegiveis = np.asarray(elegiveis)
    n_afetados = int(elegiveis.sum())
    if n_afetados == 0:
        return None

    novas_probas_elegiveis = prever_probabilidades(
        df_sim.loc[elegiveis], modelo, mm, colunas_referencia
    )

    probas_finais = probas_atuais.copy()
    probas_finais[elegiveis] = novas_probas_elegiveis

    taxa_antes = probas_atuais.mean() * 100
    taxa_depois = probas_finais.mean() * 100
    reducao_pp = taxa_antes - taxa_depois

    # Redução também olhando só para o público afetado (mais intuitivo)
    taxa_antes_grupo = probas_atuais[elegiveis].mean() * 100
    taxa_depois_grupo = novas_probas_elegiveis.mean() * 100

    return {
        'Cenário': nome,
        'Clientes_Afetados': n_afetados,
        'Taxa_Churn_Base_Antes (%)': round(taxa_antes, 2),
        'Taxa_Churn_Base_Depois (%)': round(taxa_depois, 2),
        'Redução_Base (p.p.)': round(reducao_pp, 2),
        'Taxa_Churn_Grupo_Afetado_Antes (%)': round(taxa_antes_grupo, 2),
        'Taxa_Churn_Grupo_Afetado_Depois (%)': round(taxa_depois_grupo, 2),
    }


# ============================================================
# MAIN
# ============================================================
def main():
    print("1/4 — Carregando dados e treinando o modelo...")
    caminho = carregar_dados_com_verificacao(CAMINHO_DADOS)
    df = carregar_dados(caminho)
    modelo, mm, colunas_referencia = treinar_modelo(df)

    df_ativos = df[df['Churn'] == 'No'].copy().reset_index(drop=True)
    probas_atuais = prever_probabilidades(df_ativos, modelo, mm, colunas_referencia)
    taxa_base = probas_atuais.mean() * 100
    print(f"\nTaxa de churn ESPERADA hoje (base ativa, {len(df_ativos)} clientes): {taxa_base:.2f}%")

    print("\n2/4 — Rodando simulações de cenários de ação...")
    resultados = []

    df_sim, eleg = cenario_migrar_contrato_anual(df_ativos)
    r = simular_cenario(
        "Migrar contrato mensal → anual\n(clientes com contrato mensal)",
        df_ativos, probas_atuais, df_sim, eleg, modelo, mm, colunas_referencia
    )
    if r: resultados.append(r)

    df_sim, eleg = cenario_trocar_forma_pagamento(df_ativos)
    r = simular_cenario(
        "Trocar cheque eletrônico → débito automático\n(clientes que pagam por cheque eletrônico)",
        df_ativos, probas_atuais, df_sim, eleg, modelo, mm, colunas_referencia
    )
    if r: resultados.append(r)

    df_sim, eleg = cenario_desconto_fibra(df_ativos, percentual_desconto=0.15)
    r = simular_cenario(
        "Desconto de 15% na mensalidade\n(clientes com internet fibra óptica)",
        df_ativos, probas_atuais, df_sim, eleg, modelo, mm, colunas_referencia
    )
    if r: resultados.append(r)

    df_sim, eleg = cenario_combinado_alto_risco(df_ativos, probas_atuais, limiar_alto_risco=0.50)
    r = simular_cenario(
        "Ação combinada nos clientes de ALTO risco\n(contrato anual + débito automático)",
        df_ativos, probas_atuais, df_sim, eleg, modelo, mm, colunas_referencia
    )
    if r: resultados.append(r)

    resultados_df = pd.DataFrame(resultados)

    print("\n3/4 — Resultado das simulações:\n")
    for _, r in resultados_df.iterrows():
        nome_limpo = r['Cenário'].split('\n')[0]
        print(f"• {nome_limpo}")
        print(f"    Clientes afetados: {r['Clientes_Afetados']}")
        print(f"    Taxa de churn da base: {r['Taxa_Churn_Base_Antes (%)']}% → "
              f"{r['Taxa_Churn_Base_Depois (%)']}%  "
              f"(redução de {r['Redução_Base (p.p.)']} p.p.)")
        print()

    resultados_df.to_csv(SAIDA_CSV, index=False, encoding='utf-8-sig')
    print(f"Tabela salva em: {SAIDA_CSV}")

    print("\n4/4 — Gerando gráfico comparativo...")
    gerar_grafico(resultados_df, taxa_base)

    print(f"\nConcluído! Arquivos gerados:\n  - {SAIDA_CSV}\n  - {SAIDA_GRAFICO}")
    print("\nLembrete: estes números são uma ESTIMATIVA baseada no modelo, não uma "
          "garantia. A validação real vem de um piloto medido na prática (teste A/B).")


def gerar_grafico(resultados_df, taxa_base):
    fig, ax = plt.subplots(figsize=(10, 6))
    nomes = [c.split('\n')[0] for c in resultados_df['Cenário']]
    antes = resultados_df['Taxa_Churn_Base_Antes (%)']
    depois = resultados_df['Taxa_Churn_Base_Depois (%)']

    x = np.arange(len(nomes))
    largura = 0.35

    ax.bar(x - largura/2, antes, largura, label='Antes', color='#94A3B8')
    ax.bar(x + largura/2, depois, largura, label='Depois (simulado)', color='#2DD4BF')

    for i, (a, d) in enumerate(zip(antes, depois)):
        ax.text(i - largura/2, a + 0.3, f'{a:.1f}%', ha='center', fontsize=9)
        ax.text(i + largura/2, d + 0.3, f'{d:.1f}%', ha='center', fontsize=9, fontweight='bold')

    ax.set_ylabel('Taxa de churn esperada da base (%)')
    ax.set_title('Impacto simulado de ações de retenção na taxa de churn')
    ax.set_xticks(x)
    ax.set_xticklabels(nomes, rotation=15, ha='right', fontsize=9)
    ax.legend()
    ax.axhline(taxa_base, color='#94A3B8', linestyle='--', linewidth=0.8, alpha=0.5)
    plt.tight_layout()
    plt.savefig(SAIDA_GRAFICO, dpi=130)
    plt.close()


if __name__ == '__main__':
    main()

1/4 — Carregando dados e treinando o modelo...
Acurácia do modelo no conjunto de teste: 79.95%

Taxa de churn ESPERADA hoje (base ativa, 5174 clientes): 16.75%

2/4 — Rodando simulações de cenários de ação...

3/4 — Resultado das simulações:

• Migrar contrato mensal → anual
    Clientes afetados: 2220
    Taxa de churn da base: 16.75% → 11.71%  (redução de 5.04 p.p.)

• Trocar cheque eletrônico → débito automático
    Clientes afetados: 1294
    Taxa de churn da base: 16.75% → 16.05%  (redução de 0.7 p.p.)

• Desconto de 15% na mensalidade
    Clientes afetados: 1799
    Taxa de churn da base: 16.75% → 16.35%  (redução de 0.41 p.p.)

• Ação combinada nos clientes de ALTO risco
    Clientes afetados: 359
    Taxa de churn da base: 16.75% → 15.01%  (redução de 1.75 p.p.)

Tabela salva em: simulacao_impacto_cenarios.csv

4/4 — Gerando gráfico comparativo...

Concluído! Arquivos gerados:
  - simulacao_impacto_cenarios.csv
  - simulacao_impacto_cenarios.png

Lembrete: estes números são uma